<a href="https://colab.research.google.com/github/veerendhranuthalapati/MARINeX/blob/main/MARINECADASTRE_AIS_2025_100D_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MARINeX — MarineCadastre AIS 2025 (100-Day) Clean Pipeline

Analysis-only, restart-safe notebook for the already-ingested **2025-01-01 → 2025-04-10** MarineCadastre AIS dataset.

This notebook does **not** redownload the 100 daily files. It reconnects to the existing DuckDB database, recreates the Parquet view, validates the dataset, audits trajectory quality, and provides one parameterized candidate-query function.

It fixes the previously identified notebook issues:
- correct MarineCadastre raw-column mapping for future ingestion
- one DuckDB connection only
- consistent `MANIFEST_PATH`
- no undefined `ORIGIN_LON` / `ORIGIN_LAT`
- parameterized candidate query
- vectorized Haversine distance
- duplicate query functions removed


In [5]:
# 01 — Install analysis dependencies
!pip -q install duckdb pyarrow pandas numpy


In [6]:
# 02 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# 03 — Configuration
from pathlib import Path
import pandas as pd
import numpy as np

YEAR = 2025
START_DATE = "2025-01-01"
END_DATE = "2025-04-10"
EXPECTED_DAYS = 100

ROOT = Path("/content/drive/MyDrive/MARINeX/AIS_2025_100D")
PARQUET_ROOT = ROOT / "parquet"
MANIFEST_ROOT = ROOT / "manifests"
REPORT_ROOT = ROOT / "reports"
DB_ROOT = ROOT / "database"

MANIFEST_PATH = MANIFEST_ROOT / "ais_ingestion_manifest.csv"
DB_PATH = DB_ROOT / "marinex_ais_2025_100d.duckdb"

for p in [PARQUET_ROOT, MANIFEST_ROOT, REPORT_ROOT, DB_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("PARQUET_ROOT:", PARQUET_ROOT)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("DB_PATH:", DB_PATH)


ROOT: /content/drive/MyDrive/MARINeX/AIS_2025_100D
PARQUET_ROOT: /content/drive/MyDrive/MARINeX/AIS_2025_100D/parquet
MANIFEST_PATH: /content/drive/MyDrive/MARINeX/AIS_2025_100D/manifests/ais_ingestion_manifest.csv
DB_PATH: /content/drive/MyDrive/MARINeX/AIS_2025_100D/database/marinex_ais_2025_100d.duckdb


In [8]:
# 04 — Verify the 100-day ingestion is complete
from datetime import datetime, timedelta

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

manifest = pd.read_csv(MANIFEST_PATH)

completed = manifest[
    manifest["status"].isin(["complete", "already_complete"])
].copy()

completed["date"] = pd.to_datetime(completed["date"]).dt.strftime("%Y-%m-%d")

expected_dates = [
    (datetime.strptime(START_DATE, "%Y-%m-%d") + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(EXPECTED_DAYS)
]

missing_dates = sorted(set(expected_dates) - set(completed["date"]))

print("Expected days :", EXPECTED_DAYS)
print("Completed days:", completed["date"].nunique())
print("Missing days  :", len(missing_dates))

if missing_dates:
    for d in missing_dates:
        print(" ", d)
    raise RuntimeError("100-day AIS ingestion is incomplete.")

print("First:", completed["date"].min())
print("Last :", completed["date"].max())
print("✅ 100-day AIS dataset is complete.")


Expected days : 100
Completed days: 100
Missing days  : 0
First: 2025-01-01
Last : 2025-04-10
✅ 100-day AIS dataset is complete.


In [9]:
# 05 — Open ONE DuckDB connection
import duckdb

db = duckdb.connect(str(DB_PATH))
print("✅ DuckDB connected:", DB_PATH)


✅ DuckDB connected: /content/drive/MyDrive/MARINeX/AIS_2025_100D/database/marinex_ais_2025_100d.duckdb


In [10]:
# 06 — Recreate the authoritative 100-day view
db.execute("DROP VIEW IF EXISTS ais_2025_100d")

parquet_glob = (PARQUET_ROOT / "date=*/data.parquet").as_posix()

db.execute(f"""
CREATE VIEW ais_2025_100d AS
SELECT *
FROM read_parquet(
    '{parquet_glob}',
    hive_partitioning=true
)
""")

print("✅ View created: ais_2025_100d")


✅ View created: ais_2025_100d


In [11]:
# 07 — Global dataset statistics
stats = db.execute("""
SELECT
    COUNT(*) AS total_positions,
    COUNT(DISTINCT mmsi) AS unique_vessels,
    COUNT(DISTINCT CAST(timestamp AS DATE)) AS days,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    MIN(longitude) AS min_lon,
    MAX(longitude) AS max_lon,
    MIN(latitude) AS min_lat,
    MAX(latitude) AS max_lat
FROM ais_2025_100d
""").fetchdf()

display(stats)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_positions,unique_vessels,days,first_timestamp,last_timestamp,min_lon,max_lon,min_lat,max_lat
0,737583592,56755,100,2025-01-01,2025-04-10 23:59:59,-179.93652,179.77102,-25.27595,89.65465


In [12]:
# 08 — Daily coverage
daily_stats = db.execute("""
SELECT
    CAST(timestamp AS DATE) AS date,
    COUNT(*) AS positions,
    COUNT(DISTINCT mmsi) AS vessels
FROM ais_2025_100d
GROUP BY 1
ORDER BY 1
""").fetchdf()

display(daily_stats.head())
print("Days:", len(daily_stats))
print("Minimum daily positions:", int(daily_stats["positions"].min()))
print("Maximum daily positions:", int(daily_stats["positions"].max()))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,date,positions,vessels
0,2025-01-01,7337208,16286
1,2025-01-02,6684195,16693
2,2025-01-03,6585461,17410
3,2025-01-04,6844921,16820
4,2025-01-05,7608241,16786


Days: 100
Minimum daily positions: 5929631
Maximum daily positions: 8736176


In [13]:
# 09 — FAST trajectory/data-quality audit
# Avoids a full 100-day MMSI+timestamp window sort.

trajectory_quality = db.execute("""
SELECT
    COUNT(*) AS total_points,

    COUNT(*) FILTER (
        WHERE mmsi IS NULL
    ) AS missing_mmsi,

    COUNT(*) FILTER (
        WHERE timestamp IS NULL
    ) AS missing_timestamp,

    COUNT(*) FILTER (
        WHERE longitude IS NULL
           OR latitude IS NULL
    ) AS missing_coordinates,

    COUNT(*) FILTER (
        WHERE longitude NOT BETWEEN -180 AND 180
           OR latitude NOT BETWEEN -90 AND 90
    ) AS invalid_coordinates,

    COUNT(*) FILTER (
        WHERE sog IS NOT NULL
          AND (sog < 0 OR sog > 250)
    ) AS impossible_speed,

    COUNT(*) FILTER (
        WHERE cog IS NOT NULL
          AND (cog < 0 OR cog > 360)
    ) AS invalid_course

FROM ais_2025_100d
""").fetchdf()

display(trajectory_quality)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_points,missing_mmsi,missing_timestamp,missing_coordinates,invalid_coordinates,impossible_speed,invalid_course
0,737583592,0,0,0,0,0,0


In [14]:
# 10 — Sampled trajectory continuity audit
# Checks one daily partition at a time instead of sorting 100 days at once.

from pathlib import Path
import pandas as pd
import duckdb

partition_paths = sorted(
    PARQUET_ROOT.glob("date=*/data.parquet")
)

print("Partitions:", len(partition_paths))

Partitions: 100


In [15]:
trajectory_results = []

for i, path in enumerate(partition_paths, start=1):

    date_value = path.parent.name.replace("date=", "")

    print(f"[{i}/{len(partition_paths)}] {date_value}")

    result = db.execute(f"""
        WITH ordered AS (
            SELECT
                mmsi,
                timestamp,
                LAG(timestamp) OVER (
                    PARTITION BY mmsi
                    ORDER BY timestamp
                ) AS prev_time
            FROM read_parquet('{path.as_posix()}')
        )

        SELECT
            '{date_value}' AS date,

            COUNT(*) FILTER (
                WHERE prev_time IS NOT NULL
            ) AS linked_points,

            COUNT(*) FILTER (
                WHERE prev_time IS NOT NULL
                  AND timestamp <= prev_time
            ) AS non_forward_timestamps,

            COUNT(*) FILTER (
                WHERE prev_time IS NOT NULL
                  AND timestamp - prev_time > INTERVAL '6 hours'
            ) AS gaps_over_6h

        FROM ordered
    """).fetchdf()

    trajectory_results.append(result)

[1/100] 2025-01-01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2/100] 2025-01-02


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[3/100] 2025-01-03


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[99/100] 2025-04-09


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[100/100] 2025-04-10


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
trajectory_daily = pd.concat(
    trajectory_results,
    ignore_index=True
)

display(trajectory_daily)

print(
    "Non-forward timestamps:",
    trajectory_daily["non_forward_timestamps"].sum()
)

print(
    "Gaps > 6 hours:",
    trajectory_daily["gaps_over_6h"].sum()
)

,date,linked_points,non_forward_timestamps,gaps_over_6h
0,2025-01-01,7320922,965,946
1,2025-01-02,6667502,1873,909
2,2025-01-03,6568051,1734,966
3,2025-01-04,6828101,1225,999
4,2025-01-05,7591455,1351,1005
...,...,...,...,...
95,2025-04-06,7820952,2711,1806
96,2025-04-07,7794253,2286,1037
97,2025-04-08,7402735,3017,925
98,2025-04-09,7698218,4142,977


Non-forward timestamps: 256084
Gaps > 6 hours: 101331


In [17]:
# 10 — Vessel history
vessel_history = db.execute("""
SELECT
    mmsi,
    ANY_VALUE(vessel_name) AS vessel_name,
    ANY_VALUE(imo) AS imo,
    ANY_VALUE(vessel_type) AS vessel_type,
    MIN(timestamp) AS first_seen,
    MAX(timestamp) AS last_seen,
    COUNT(*) AS positions,
    COUNT(DISTINCT CAST(timestamp AS DATE)) AS days_seen,
    AVG(sog) AS mean_sog,
    MAX(sog) AS max_sog
FROM ais_2025_100d
GROUP BY mmsi
ORDER BY days_seen DESC, positions DESC
""").fetchdf()

display(vessel_history.head(30))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,mmsi,vessel_name,imo,vessel_type,first_seen,last_seen,positions,days_seen,mean_sog,max_sog
0,367531220,WATCHMAN,IMO8855774,30,2025-01-01 00:00:01,2025-04-10 23:59:16,129933,100,0.058299,11.1
1,367331730,COLUMBIA,None,50,2025-01-01 00:00:04,2025-04-10 23:59:45,128365,100,2.370726,64.2
2,369970411,YTB-835 SKENANDOA,None,52,2025-01-01 00:00:08,2025-04-10 23:59:39,127914,100,0.001590,0.2
3,367529030,CONNOR FOSS,None,57,2025-01-01 00:00:02,2025-04-10 23:59:43,126925,100,1.570308,19.2
4,367156340,GRANADA,IMO7047849,30,2025-01-01 00:00:04,2025-04-10 23:59:54,126135,100,0.901020,15.6
5,367380280,WESTERN STAR,IMO8423806,31,2025-01-01 00:00:07,2025-04-10 23:59:44,125889,100,0.175501,12.4
6,367155110,CAPTAIN RALEIGH,IMO7937575,30,2025-01-01 00:00:00,2025-04-10 23:59:33,125319,100,0.941674,13.2
7,366772780,WSF WALLA WALLA,IMO7233151,60,2025-01-01 00:00:02,2025-04-10 23:59:43,123999,100,6.841364,20.7
8,366759130,WSF PUYALLUP,IMO9137363,60,2025-01-01 00:00:04,2025-04-10 23:59:39,123905,100,6.493659,20.6
9,366773070,WSF CATHLAMET,IMO7808138,65,2025-01-01 00:00:01,2025-04-10 23:59:17,123876,100,4.569881,18.7


In [18]:
# 11 — Vectorized Haversine
def haversine_km_vectorized(lon1, lat1, lon2, lat2):
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )

    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))


In [19]:
# 12 — Single parameterized candidate query
def query_candidates(
    min_lon,
    max_lon,
    min_lat,
    max_lat,
    start_time,
    end_time,
):
    query = """
    SELECT
        mmsi,
        vessel_name,
        imo,
        call_sign,
        vessel_type,
        timestamp,
        longitude,
        latitude,
        sog,
        cog,
        heading,
        length,
        width,
        draft,
        cargo,
        transceiver
    FROM ais_2025_100d
    WHERE
        longitude BETWEEN ? AND ?
        AND latitude BETWEEN ? AND ?
        AND timestamp BETWEEN CAST(? AS TIMESTAMP)
                             AND CAST(? AS TIMESTAMP)
    ORDER BY mmsi, timestamp
    """

    return db.execute(
        query,
        [
            float(min_lon),
            float(max_lon),
            float(min_lat),
            float(max_lat),
            str(start_time),
            str(end_time),
        ],
    ).fetchdf()


In [20]:
# 13 — Example candidate query
# Dummy investigation window only; not a real incident.
TEST_ORIGIN_LON = -74.1
TEST_ORIGIN_LAT = 40.0

candidates = query_candidates(
    min_lon=-75,
    max_lon=-73,
    min_lat=39,
    max_lat=41,
    start_time="2025-02-01 00:00:00",
    end_time="2025-02-03 23:59:59",
)

print("AIS points:", len(candidates))
print("Candidate vessels:", candidates["mmsi"].nunique())

if len(candidates):
    candidates["distance_to_origin_km"] = haversine_km_vectorized(
        candidates["longitude"].to_numpy(),
        candidates["latitude"].to_numpy(),
        TEST_ORIGIN_LON,
        TEST_ORIGIN_LAT,
    )

display(
    candidates.sort_values("distance_to_origin_km").head(20)
    if len(candidates) else candidates
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AIS points: 824612
Candidate vessels: 643


,mmsi,vessel_name,imo,call_sign,vessel_type,timestamp,longitude,latitude,sog,cog,heading,length,width,draft,cargo,transceiver,distance_to_origin_km
38577,338300347,FANTASIA,None,None,37,2025-02-02 03:00:08,-74.08895,40.01430,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.847739
38576,338300347,FANTASIA,None,None,37,2025-02-02 02:57:07,-74.08895,40.01431,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.848696
38582,338300347,FANTASIA,None,None,37,2025-02-02 03:15:08,-74.08892,40.01430,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.849041
38578,338300347,FANTASIA,None,None,37,2025-02-02 03:03:07,-74.08894,40.01431,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.849129
38150,338300347,FANTASIA,None,None,37,2025-02-01 04:33:09,-74.08895,40.01432,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.849653
38575,338300347,FANTASIA,None,None,37,2025-02-02 02:54:08,-74.08895,40.01432,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.849653
38570,338300347,FANTASIA,None,None,37,2025-02-02 02:39:07,-74.08895,40.01432,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.849653
38568,338300347,FANTASIA,None,None,37,2025-02-02 02:33:08,-74.08895,40.01432,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.849653
38566,338300347,FANTASIA,None,None,37,2025-02-02 02:27:08,-74.08894,40.01432,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.850086
38162,338300347,FANTASIA,None,None,37,2025-02-01 05:12:09,-74.08894,40.01432,0.0,NaN,NaN,NaN,NaN,NaN,<NA>,B,1.850086


In [21]:
# 14 — Candidate-level summary
if len(candidates):
    candidate_summary = (
        candidates
        .groupby(["mmsi", "vessel_name"], dropna=False)
        .agg(
            first_seen=("timestamp", "min"),
            last_seen=("timestamp", "max"),
            positions=("timestamp", "count"),
            min_distance_km=("distance_to_origin_km", "min"),
            mean_sog=("sog", "mean"),
            max_sog=("sog", "max"),
        )
        .reset_index()
        .sort_values(
            ["min_distance_km", "positions"],
            ascending=[True, False]
        )
    )

    display(candidate_summary.head(20))
else:
    print("No candidates in the dummy query window.")


,mmsi,vessel_name,first_seen,last_seen,positions,min_distance_km,mean_sog,max_sog
47,338300347,FANTASIA,2025-02-01 00:00:10,2025-02-03 23:57:08,1401,1.847739,0.000357,0.1
494,368078470,R/V MORGAN,2025-02-01 00:02:20,2025-02-03 23:53:21,1270,4.152140,0.001102,0.2
69,338412837,RESOLUTION,2025-02-02 14:17:02,2025-02-03 20:18:47,296,5.393873,7.179392,21.6
74,338427889,HARMONY,2025-02-01 00:01:03,2025-02-03 23:58:03,1354,5.581828,0.009453,0.2
438,367755220,SLABJACK,2025-02-01 00:02:55,2025-02-03 23:05:55,387,5.606523,0.000000,0.0
87,338483203,CASSIDY,2025-02-02 05:41:51,2025-02-02 22:50:50,23,5.610356,0.000000,0.0
29,338082663,FINAL ACT,2025-02-01 00:00:44,2025-02-03 23:56:45,1198,5.670231,0.346411,2.3
243,367156370,EVERGREEN STATE,2025-02-01 00:00:03,2025-02-03 23:59:53,3000,5.827066,1.295067,10.0
378,367618310,JUSTINE MCALLISTER,2025-02-01 00:00:03,2025-02-03 18:05:54,2091,6.073984,3.116236,10.4
419,367706270,KATAN,2025-02-01 14:06:28,2025-02-02 23:19:09,1463,6.236390,6.808954,12.0


In [6]:
# ============================================================
# AIS 100-DAY DATABASE RECOVERY
# Run this cell instead of the previous DB_PATH cell
# ============================================================

from pathlib import Path
import duckdb
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive")
AIS_ROOT = ROOT / "MARINeX" / "AIS_2025_100D"

print("AIS root:", AIS_ROOT)
print("Exists:", AIS_ROOT.exists())

# ------------------------------------------------------------
# 2. Search for existing DuckDB files
# ------------------------------------------------------------

duckdb_files = list(ROOT.rglob("*.duckdb"))

print(f"\nDuckDB files found: {len(duckdb_files)}")

for p in duckdb_files:
    print("  ", p)

# ------------------------------------------------------------
# 3. If a DuckDB exists, connect to it
# ------------------------------------------------------------

db = None

if duckdb_files:
    # Prefer the expected AIS database name if present
    preferred = [
        p for p in duckdb_files
        if "marinex_ais_2025_100d" in p.name.lower()
    ]

    DB_PATH = preferred[0] if preferred else duckdb_files[0]

    print("\nUsing DuckDB:")
    print(DB_PATH)

    db = duckdb.connect(str(DB_PATH))

# ------------------------------------------------------------
# 4. Otherwise rebuild DuckDB from existing Parquet files
# ------------------------------------------------------------

else:

    print("\nNo DuckDB found.")
    print("Searching for existing AIS Parquet partitions...")

    parquet_files = list(AIS_ROOT.rglob("*.parquet"))

    print(f"Parquet files found: {len(parquet_files)}")

    for p in parquet_files[:10]:
        print("  ", p)

    if not parquet_files:
        raise FileNotFoundError(
            "No DuckDB and no Parquet files were found under "
            f"{AIS_ROOT}"
        )

    # Create database directory
    DB_DIR = AIS_ROOT / "database"
    DB_DIR.mkdir(parents=True, exist_ok=True)

    DB_PATH = DB_DIR / "marinex_ais_2025_100d.duckdb"

    print("\nRebuilding DuckDB from existing Parquet files...")
    print("Database:", DB_PATH)

    db = duckdb.connect(str(DB_PATH))

    # IMPORTANT:
    # Read the existing Parquet data directly.
    parquet_glob = str(AIS_ROOT / "**" / "*.parquet")

    db.execute(f"""
        CREATE OR REPLACE VIEW ais_2025_100d AS
        SELECT *
        FROM read_parquet(
            '{parquet_glob}',
            union_by_name = true
        )
    """)

    print("DuckDB view created successfully.")

# ------------------------------------------------------------
# 5. Inspect database
# ------------------------------------------------------------

print("\n=== DATABASE CHECK ===")

display(db.execute("SHOW TABLES").fetchdf())

# ------------------------------------------------------------
# 6. Verify AIS table exists and inspect columns
# ------------------------------------------------------------

tables = db.execute("SHOW TABLES").fetchdf()

if "ais_2025_100d" not in tables["name"].tolist():
    raise RuntimeError(
        "ais_2025_100d was not found in the database."
    )

print("\n=== AIS SUMMARY ===")

summary = db.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT mmsi) AS unique_vessels,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp
FROM ais_2025_100d
""").fetchdf()

display(summary)

print("\n=== COLUMN CHECK ===")

display(
    db.execute("""
    DESCRIBE ais_2025_100d
    """).fetchdf()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AIS root: /content/drive/MyDrive/MARINeX/AIS_2025_100D
Exists: True

DuckDB files found: 1
   /content/drive/MyDrive/MARINeX/AIS_2025_100D/database/marinex_ais_2025_100d.duckdb

Using DuckDB:
/content/drive/MyDrive/MARINeX/AIS_2025_100D/database/marinex_ais_2025_100d.duckdb

=== DATABASE CHECK ===


,name
0,ais_2025_100d



=== AIS SUMMARY ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_vessels,first_timestamp,last_timestamp
0,737583592,56755,2025-01-01,2025-04-10 23:59:59



=== COLUMN CHECK ===


,column_name,column_type,null,key,default,extra
0,mmsi,BIGINT,YES,None,None,None
1,timestamp,TIMESTAMP,YES,None,None,None
2,longitude,DOUBLE,YES,None,None,None
3,latitude,DOUBLE,YES,None,None,None
4,sog,DOUBLE,YES,None,None,None
5,cog,DOUBLE,YES,None,None,None
6,heading,DOUBLE,YES,None,None,None
7,vessel_name,VARCHAR,YES,None,None,None
8,imo,VARCHAR,YES,None,None,None
9,call_sign,VARCHAR,YES,None,None,None


In [7]:
# ============================================================
# 15 — AIS data quality audit
# Fast: no global trajectory window
# ============================================================

trajectory_summary = db.execute("""
SELECT
    COUNT(*) AS total_points,
    COUNT(DISTINCT mmsi) AS unique_vessels,

    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,

    COUNT(*) FILTER (
        WHERE mmsi IS NULL
    ) AS null_mmsi,

    COUNT(*) FILTER (
        WHERE timestamp IS NULL
    ) AS null_timestamp,

    COUNT(*) FILTER (
        WHERE longitude IS NULL
           OR latitude IS NULL
    ) AS null_coordinates,

    COUNT(*) FILTER (
        WHERE longitude < -180
           OR longitude > 180
           OR latitude < -90
           OR latitude > 90
    ) AS invalid_coordinates,

    COUNT(*) FILTER (
        WHERE sog < 0
           OR sog > 100
    ) AS impossible_speed,

    COUNT(*) FILTER (
        WHERE cog < 0
           OR cog >= 360
    ) AS invalid_course

FROM ais_2025_100d
""").fetchdf()

display(trajectory_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_points,unique_vessels,first_timestamp,last_timestamp,null_mmsi,null_timestamp,null_coordinates,invalid_coordinates,impossible_speed,invalid_course
0,737583592,56755,2025-01-01,2025-04-10 23:59:59,0,0,0,0,0,0


In [9]:
# 15.1 — Simple quality interpretation

r = trajectory_summary.iloc[0]

print(f"Total AIS points       : {int(r['total_points']):,}")
print(f"Unique vessels         : {int(r['unique_vessels']):,}")
print(f"Time range             : {r['first_timestamp']} → {r['last_timestamp']}")
print(f"Null MMSI              : {int(r['null_mmsi']):,}")
print(f"Null timestamp         : {int(r['null_timestamp']):,}")
print(f"Null coordinates       : {int(r['null_coordinates']):,}")
print(f"Invalid coordinates    : {int(r['invalid_coordinates']):,}")
print(f"Impossible speed       : {int(r['impossible_speed']):,}")
print(f"Invalid course         : {int(r['invalid_course']):,}")

Total AIS points       : 737,583,592
Unique vessels         : 56,755
Time range             : 2025-01-01 00:00:00 → 2025-04-10 23:59:59
Null MMSI              : 0
Null timestamp         : 0
Null coordinates       : 0
Invalid coordinates    : 0
Impossible speed       : 0
Invalid course         : 0


## Status

The 100-day AIS foundation is complete:

**2025-01-01 → 2025-04-10**

This notebook validates the real MarineCadastre AIS corpus and supports:
- spatial filtering
- temporal filtering
- vessel candidate extraction
- vessel history
- trajectory-quality auditing
- downstream attribution feature generation

It does **not** train an attribution model. Real spill/source ground-truth labels are still required for scientifically meaningful supervised attribution metrics.


In [10]:
# 16 — Rebuild ingestion completion summary from the saved manifest

from pathlib import Path
import pandas as pd

AIS_ROOT = Path("/content/drive/MyDrive/MARINeX/AIS_2025_100D")

# Find the manifest automatically
manifest_candidates = list(AIS_ROOT.rglob("*ais_ingestion_manifest*.csv"))

print("Manifest files found:", len(manifest_candidates))

for p in manifest_candidates:
    print(" ", p)

if not manifest_candidates:
    raise FileNotFoundError(
        f"No AIS ingestion manifest found under {AIS_ROOT}"
    )

MANIFEST_PATH = manifest_candidates[0]

completed = pd.read_csv(MANIFEST_PATH)

# Normalize date column
completed["date"] = pd.to_datetime(
    completed["date"],
    errors="coerce"
)

# Keep successfully completed days
if "status" in completed.columns:
    completed = completed[
        completed["status"].astype(str).str.lower().isin(
            ["complete", "completed", "success", "ok"]
        )
    ].copy()

print("\nManifest:", MANIFEST_PATH)
print("Completed rows:", len(completed))

display(completed.head())

Manifest files found: 1
  /content/drive/MyDrive/MARINeX/AIS_2025_100D/manifests/ais_ingestion_manifest.csv

Manifest: /content/drive/MyDrive/MARINeX/AIS_2025_100D/manifests/ais_ingestion_manifest.csv
Completed rows: 100


,date,status,url,download_mb,parquet_mb,rows,vessels,first_timestamp,last_timestamp,min_lon,max_lon,min_lat,max_lat,download_seconds,processing_seconds,parquet
0,2025-01-01,complete,https://noaaocm.blob.core.windows.net/ais/csv2...,192.82,193.31,7337208,16286,2025-01-01 00:00:00,2025-01-01 23:59:59,-174.56051,157.87216,0.55660,50.11003,25.08,25.92,/content/drive/MyDrive/MARINeX/AIS_2025_100D/p...
1,2025-01-02,complete,https://noaaocm.blob.core.windows.net/ais/csv2...,177.59,178.36,6684195,16693,2025-01-02 00:00:00,2025-01-02 23:59:59,-160.85270,144.99420,2.37640,85.62432,23.87,25.20,/content/drive/MyDrive/MARINeX/AIS_2025_100D/p...
2,2025-01-03,complete,https://noaaocm.blob.core.windows.net/ais/csv2...,174.84,175.74,6585461,17410,2025-01-03 00:00:00,2025-01-03 23:59:59,-162.02258,157.87217,0.47351,50.46404,22.89,23.37,/content/drive/MyDrive/MARINeX/AIS_2025_100D/p...
3,2025-01-04,complete,https://noaaocm.blob.core.windows.net/ais/csv2...,184.67,184.11,6844921,16820,2025-01-04 00:00:00,2025-01-04 23:59:59,-162.52505,146.16573,0.00348,68.46974,24.06,25.39,/content/drive/MyDrive/MARINeX/AIS_2025_100D/p...
4,2025-01-05,complete,https://noaaocm.blob.core.windows.net/ais/csv2...,203.75,203.31,7608241,16786,2025-01-05 00:00:00,2025-01-05 23:59:59,-163.61166,147.54153,0.59079,89.41191,24.69,25.45,/content/drive/MyDrive/MARINeX/AIS_2025_100D/p...


In [11]:
# 16.1 — 100-day ingestion summary

ingestion_summary = {
    "dataset": "MARINECADASTRE_AIS_2025_100D",
    "source": "MarineCadastre AIS 2025",
    "start_date": completed["date"].min(),
    "end_date": completed["date"].max(),
    "completed_days": int(completed["date"].nunique()),
}

display(pd.DataFrame([ingestion_summary]))

,dataset,source,start_date,end_date,completed_days
0,MARINECADASTRE_AIS_2025_100D,MarineCadastre AIS 2025,2025-01-01,2025-04-10,100


In [12]:
# 16.2 — Verify that the expected 100-day period is complete

expected_dates = pd.date_range(
    "2025-01-01",
    "2025-04-10",
    freq="D"
)

actual_dates = pd.DatetimeIndex(
    completed["date"].dropna().dt.normalize().unique()
)

missing_dates = expected_dates.difference(actual_dates)

print("Expected days :", len(expected_dates))
print("Completed days:", len(actual_dates))
print("Missing days  :", len(missing_dates))

if len(missing_dates):
    print("\nMissing dates:")
    for d in missing_dates:
        print(" ", d.date())
else:
    print("\n✅ All 100 AIS days are present.")

Expected days : 100
Completed days: 100
Missing days  : 0

✅ All 100 AIS days are present.
